In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import clip
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import pandas as pd
import numpy as np
import os

In [2]:
# 設置設備
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")

clip_model, preprocess = clip.load("RN101", device=device)

for param in clip_model.parameters():
    param.requires_grad = False

print(clip_model.visual)

Using cuda


100%|███████████████████████████████████████| 278M/278M [02:26<00:00, 2.00MiB/s]


ModifiedResNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU(inplace=True)
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU(inplace=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu3): ReLU(inplace=True)
  (avgpool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), str

In [10]:
class CLIP_EMNIST_Model(nn.Module):
    def __init__(self, clip_model, num_classes=62):
        super(CLIP_EMNIST_Model, self).__init__()
        self.clip_visual = clip_model.visual  # CLIP 的影像 encoder
        self.fc = nn.Sequential(
            nn.Linear(1024, 256),  # RN101 的輸出是 1024 維
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)  # 分類 62 個類別
        )

    def forward(self, x):
        with torch.no_grad():  # 凍結 CLIP
            features = self.clip_visual(x)
        return self.fc(features)

In [4]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # CLIP 需要 RGB
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # 測試集相同處理
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [5]:
class EMNISTDataset(Dataset):
    def __init__(self, npz_path, transform=None, has_labels=True):
        # 讀取 npz 檔案
        data = np.load(npz_path)
        
        # 提取影像並去掉最後一個通道維度
        self.images = data["training_images" if has_labels else "testing_images"]  
        #self.images = self.images.squeeze(-1).astype(np.uint8)   
                # 根據影像維度進行調整
        if len(self.images.shape) > 3:
            # 如果有額外的維度 (例如通道維度)，進行 squeeze
            self.images = self.images.squeeze(-1)
        
        # 檢查數據範圍並歸一化
        if self.images.max() <= 1.0 and self.images.min() >= 0:
            # 如果數據範圍是 [0, 1]，轉換為 [0, 255]
            self.images = (self.images * 255).astype(np.uint8)
        elif self.images.max() > 1.0 and self.images.max() <= 255:
            # 如果數據已在 [0, 255] 範圍內，確保類型為 uint8
            self.images = self.images.astype(np.uint8)
        else:
            # 如果數據範圍異常，進行最小最大值歸一化
            self.images = ((self.images - self.images.min()) / 
                         (self.images.max() - self.images.min() + 1e-8) * 255).astype(np.uint8)
        

        # 提取標籤（如果有）
        self.labels = data["training_labels"] if has_labels else None
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # 轉換為 PIL 影像
        image = Image.fromarray(self.images[idx])

        if self.transform:
            image = self.transform(image)

        if self.has_labels:
            label = int(self.labels[idx].item())  # 轉換為純量
            return image, label
        
        return image,  # 注意這裡要回傳 tuple

In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt

# 讀取數據
train_dataset = EMNISTDataset("ntutemnist_data/emnist-byclass-train.npz", transform=train_transform)
test_dataset = EMNISTDataset("ntutemnist_data/emnist-byclass-test.npz", transform=test_transform, has_labels=False)


# 切割 10% 訓練集作為驗證集
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

# DataLoader
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)


#================================================ 檢查 ==============================================
# 檢查圖片轉換後的大小
def check_image_size(loader):
    # 從 DataLoader 中取出一個 batch
    images, _ = next(iter(loader))  # 這裡假設 loader 會返回 image 和 label
    print(f"Image batch shape: {images.shape}")  # 打印圖片的 shape

# 檢查訓練集轉換後的圖片大小
check_image_size(train_dataset)

# 檢查驗證集轉換後的圖片大小
check_image_size(val_loader)


def show_images(loader, num_images=5):
    # 從 DataLoader 中取出一個 batch
    images, labels = next(iter(loader))  # 假設 loader 會返回 image 和 label
    
    # 轉換為 numpy 陣列以便使用 matplotlib 顯示
    images = images.numpy()  # 這是將 pytorch tensor 轉換為 numpy 陣列
    
    # 顯示前 num_images 張圖片
    plt.figure(figsize=(10, 10))
    for i in range(num_images):
        plt.subplot(1, num_images, i+1)
        img = images[i].transpose(1, 2, 0)  # 轉換維度為 (H, W, C)
        plt.imshow(img.squeeze(), cmap='gray')  # 轉換為灰階圖片（若是單通道）
        plt.title(f"Label: {labels[i].item()}")
        plt.axis("off")
    plt.show()

# 顯示訓練集前 5 張圖片
show_images(train_loader, num_images=5)
show_images(val_loader, num_images=5)

#=================================================================================================

# 初始化模型
model = CLIP_EMNIST_Model(clip_model).to(torch.float32).to(device)
print(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)

from tqdm import tqdm  # 引入 tqdm

# 訓練函數
def train(model, train_loader, val_loader, epochs, save_path="model.pth"):
    best_val_acc = 0.0  # 用來追蹤最佳驗證準確率
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        # 計算訓練與驗證準確度
        train_acc = correct / total
        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}/{epochs}: Loss={total_loss/len(train_loader):.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

        # 如果目前的模型比之前的最佳驗證準確率更高，就儲存權重
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"🔽 New best model saved at epoch {epoch+1} with Val Acc={val_acc:.4f}")

def evaluate(model, val_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 開始訓練
# 訓練模型並儲存權重
train(model, train_loader, val_loader, epochs=30, save_path="best_model.pth")